# SAM Box-Prompted Fine-Tune — Kaggle GPU Training

Companion to `kaggle/bc_tumor_segmentation_kaggle.ipynb` (that one trains the
from-scratch U-Net end-to-end; this one fine-tunes SAM's mask decoder only,
given a box prompt derived from the ground-truth mask - it does NOT decide
tumor presence, see `configs/sam_finetune.yaml`'s header comment). Full
write-up: `SAM_WALKTHROUGH.md`. Design rationale: `improving_model.md` Tier 5.

Research prototype — **not a clinical diagnostic system**. Repo:
[https://github.com/ahmad-pervaiz/Breast-Cancer-Detection-3D-Model](https://github.com/ahmad-pervaiz/Breast-Cancer-Detection-3D-Model)

**Before running:** Settings (right panel) →
- Accelerator: **GPU T4 x2** (or P100)
- Internet: **On** (needed to `git clone`/`pull` the repo, `pip install segment-anything`, and download the SAM checkpoint)
- Add Data → attach your uploaded `breast_tumor_dataset.zip` Kaggle Dataset (the same one the U-Net notebook uses)


## 1. Environment check — confirm GPU is actually attached

In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| CUDA version:", torch.version.cuda)
else:
    print("\n*** WARNING: no GPU detected. ***")
    print("Go to Settings (right panel) -> Accelerator -> GPU T4 x2 (or P100), then Save and re-run this cell.")
    print("Training will proceed on CPU if you continue, but will take hours instead of minutes.")


## 2. Clone (or update) the project code from GitHub

**`BRANCH`** below controls which branch gets checked out — change it to test
an experiment branch without touching `main`. Re-running this cell in an
existing session fetches and re-checks-out `BRANCH` instead of reusing a
possibly-stale clone.

If you didn't push to GitHub yet, see `WALKTHROUGH.md` section 1.


In [ ]:
import os, subprocess

REPO_URL = "https://github.com/ahmad-pervaiz/Breast-Cancer-Detection-3D-Model.git"
BRANCH = "main"  # change to e.g. "experiment/lr-schedule-tuning" to test a branch
# The GitHub repo root IS this project (configs/, src/, scripts/ live at repo
# root) - there is no nested bc_tumor_detection/ subfolder inside the clone.
PROJECT_DIR = "/kaggle/working/Breast-Cancer-Detection-3D-Model"

if not os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
else:
    # Already cloned earlier in this session - fetch instead of reusing a stale copy.
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin"], check=True)

subprocess.run(["git", "-C", PROJECT_DIR, "checkout", BRANCH], check=True)
subprocess.run(["git", "-C", PROJECT_DIR, "pull", "origin", BRANCH], check=True)

if not os.path.exists(os.path.join(PROJECT_DIR, "configs", "config.yaml")):
    raise RuntimeError(
        f"{PROJECT_DIR}/configs/config.yaml is missing after clone/checkout - "
        f"check branch '{BRANCH}' actually contains this project at its root: {REPO_URL}"
    )

os.chdir(PROJECT_DIR)
print(f"cwd: {os.getcwd()}  (branch: {BRANCH})")


In [ ]:
!ls

## 3. Install dependencies

Kaggle's base image already has PyTorch+CUDA, pandas, numpy, matplotlib,
scipy, scikit-image. `segment-anything` (the SAM library itself) is the one
new thing this notebook needs beyond what the U-Net notebook installs.

In [ ]:
!pip install -q albumentations opencv-python-headless pyyaml segment-anything
print("done")


## 4. Locate the attached dataset

In [ ]:
from pathlib import Path

REQUIRED_SUBDIRS = ["Train", "Train_masks", "valid", "valid_masks", "test"]
candidates = [
    p for p in Path("/kaggle/input").iterdir()
    if p.is_dir() and all((p / sub).exists() for sub in REQUIRED_SUBDIRS)
]
if not candidates:
    raise RuntimeError(
        "No attached dataset with Train/Train_masks/valid/valid_masks/test folders found under "
        "/kaggle/input. Add Data -> attach your breast_tumor_dataset.zip Kaggle Dataset, then re-run this cell."
    )
if len(candidates) > 1:
    print(f"WARNING: multiple matching datasets found, using the first: {candidates}")
DATASET_ROOT = str(candidates[0])
print("Using dataset_root:", DATASET_ROOT)
!ls "{DATASET_ROOT}"


## 5. Get a SAM checkpoint

Two options:

**A. Vanilla SAM ViT-B (default, no manual step)** — Meta's original
weights, pretrained on natural images only. Downloaded directly below.

**B. MedSAM ViT-B (recommended — already fine-tuned on medical images with
box prompts, see [github.com/bowang-lab/MedSAM](https://github.com/bowang-lab/MedSAM))**.
Its checkpoint is Google-Drive-hosted, not reliably scriptable — download it
yourself from that repo's README, upload it as a new Kaggle Dataset (any
name), Add Data → attach it here. The cell below auto-detects it (looks for
a `.pth` file with "medsam" in the name under `/kaggle/input`) and uses it
instead of downloading vanilla SAM.

In [ ]:
import glob
import subprocess
from pathlib import Path

CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)

medsam_candidates = glob.glob("/kaggle/input/**/*medsam*.pth", recursive=True)
if medsam_candidates:
    SAM_CHECKPOINT = medsam_candidates[0]
    print(f"Found attached MedSAM checkpoint: {SAM_CHECKPOINT}")
else:
    SAM_CHECKPOINT = str(CHECKPOINT_DIR / "sam_vit_b_01ec64.pth")
    if not Path(SAM_CHECKPOINT).exists():
        print("No MedSAM dataset attached - downloading vanilla SAM ViT-B instead "
              "(pretrained on natural images only - see the cell above for the MedSAM option).")
        subprocess.run(["wget", "-q", "-O", SAM_CHECKPOINT,
                         "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"], check=True)
    print(f"Using: {SAM_CHECKPOINT}")


## 6. Smoke test

2 epochs on a tiny subset of tumor-positive images — confirms the pipeline
runs correctly in this environment before committing GPU-hours to a full
run. SAM's ViT-B encoder is heavier per-image than the U-Net, so expect this
to take longer than the U-Net notebook's smoke test, but still well under a
few minutes on a T4.

In [ ]:
!python scripts/train_sam.py --config configs/sam_finetune.yaml --dataset_root "{DATASET_ROOT}" --sam_checkpoint "{SAM_CHECKPOINT}" --smoke_test


## 6.5. (Optional) Enable ClearML experiment tracking

Same as the U-Net notebook — add `CLEARML_API_ACCESS_KEY` / `CLEARML_API_SECRET_KEY`
as Kaggle Secrets if you want this run tracked too. Harmless to run either way.

In [ ]:
import os

clearml_enabled = False
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["CLEARML_API_ACCESS_KEY"] = secrets.get_secret("CLEARML_API_ACCESS_KEY")
    os.environ["CLEARML_API_SECRET_KEY"] = secrets.get_secret("CLEARML_API_SECRET_KEY")
    os.environ.setdefault("CLEARML_API_HOST", "https://api.clear.ml")
    os.environ.setdefault("CLEARML_WEB_HOST", "https://app.clear.ml")
    os.environ.setdefault("CLEARML_FILES_HOST", "https://files.clear.ml")
    clearml_enabled = True
    print("ClearML credentials loaded from Kaggle Secrets - tracking will be enabled below.")
except Exception as e:
    print(f"No ClearML Kaggle Secrets configured ({e}) - continuing without experiment tracking.")


## 7. Full fine-tune

Uses every setting from `configs/sam_finetune.yaml` on the checked-out
branch (box padding/jitter, learning rate, epochs, early stopping, etc.)
except `dataset_root`/`sam.checkpoint`, overridden to the Kaggle paths above.

In [ ]:
import subprocess

cmd = ["python", "scripts/train_sam.py", "--config", "configs/sam_finetune.yaml",
       "--dataset_root", DATASET_ROOT, "--sam_checkpoint", SAM_CHECKPOINT]
if clearml_enabled:
    cmd.append("--clearml")
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


## 8. View training curves + a validation prediction montage inline

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

for name in ["loss_curve.png", "dice_curve.png", "iou_curve.png"]:
    img = Image.open(f"runs/sam_segmentation/plots/{name}")
    plt.figure(figsize=(6, 4)); plt.imshow(img); plt.axis("off"); plt.title(name); plt.show()


In [ ]:
import glob
latest_montage = sorted(glob.glob("runs/sam_segmentation/predictions/validation/epoch_*.png"))[-1]
Image.open(latest_montage)


## 9. Zip results for download

Same pattern as the U-Net notebook — a single zip of checkpoints/logs/plots
is easier to grab from the Output tab than digging through the repo clone.

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/sam_segmentation_results", "zip", "runs/sam_segmentation")
print("Saved: /kaggle/working/sam_segmentation_results.zip")
print("Download it from the notebook's Output tab (top right, after committing/saving the run).")


## 10. Next steps

1. Download `sam_segmentation_results.zip` from the Output tab.
2. Unzip it locally into `bc_tumor_detection/runs/sam_segmentation/`
   (matching the layout the repo expects).
3. Compare against the U-Net baseline — see `SAM_WALKTHROUGH.md` step 5:

```
python scripts/compare_sam_vs_unet.py \
    --unet_checkpoint runs/segmentation/checkpoints/best_model.pth \
    --sam_checkpoint runs/sam_segmentation/checkpoints/best_model.pth
```

4. Update `improving_model.md`'s "Run #5" changelog entry with the real
   `val_dice`/`val_iou` numbers (same discipline as every other run there).